# Scenario 1: Customer Support Resolution Agent
## Hands-on Production Scenario Workbench
**Certification Focus**: CCAR-F Exam Scenarios 1–6 | **Standard**: Claude 4.5 & Extended Thinking

---

### 📊 Production SLAs & Non-Negotiables
- First-Contact Resolution (FCR) >= 80%
- Turnaround Latency <= 2.5s p95
- $0 unauthorized refund loss (100% deterministic gate enforcement)
- 100% immediate zero-friction transfer upon explicit human request
- Max context window <= 25,000 tokens through turn 20 via PostToolUse trimming and Case Facts

---

### 🏗️ Architecture & Decision Flow
```mermaid
flowchart TD
    UserReq["Customer Interaction Utterance"] --> Router{"Evaluate Intent & State"}
    
    Router -->|"Customer Demands Human"| Escalate["Zero-Friction Human Escalation<br>(Structured Case Facts Handoff)"]
    Router -->|"Mutative Action Unauthenticated"| GateBlock["Programmatic State Gate<br>(Reject: Authentication Required)"]
    Router -->|"Identity Verified & Valid"| ExecTool["Execute Tool<br>(Trim 50+ DB fields to 4-5 via PostToolUse)"]
    Router -->|"Multiple Matches Found"| Disambiguate["Prompt Customer for Disambiguating ID"]
```

---

### 🏆 Golden Architectural Rules Reference
| Rule | Production Implementation & Rationale |
|---|---|
| **Deterministic Prerequisite Gates (TS 1.4)** | Assert identity in code before calling payment/shipping APIs. Never rely on prompts alone. |
| **Zero-Friction Immediate Escalation (TS 5.2)** | Transfer immediately upon human request. Never probe angry users for reasons. |
| **Aggressive Tool Trimming via PostToolUse (TS 5.1)** | Strip bloated CRM payloads down to 4-5 customer-relevant attributes before saving in context. |
| **Immutable Case Facts Architecture (TS 5.1)** | Store Order ID, Customer ID, and verified status in a dedicated state block outside summaries. |
| **Disambiguation on Collisions (TS 5.2)** | When lookup returns 2+ accounts, prompt user for unique identifier. Never guess. |

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns.
- **Case Facts Preservation**: Immutable transactional entities must be maintained in a dedicated Case Facts block outside conversation summaries.



In [ ]:
# Step 1: Environment Setup & Simulation Import
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scenarios.scenario_1_customer_support.simulation import *

print(' Scenario Workbench Initialized: Scenario 1: Customer Support Resolution Agent')
print(' Zero live API calls, 100% offline determinism')


### 🧪 Step 2: Core Execution Flow & Happy Path Simulation
Demonstrating end-to-end pipeline execution with audit logging.


In [ ]:
# Execute Happy Path: Authentication + Return Lookup with Trimming
pipeline = CustomerSupportPipeline()

# 1. Auth
auth_res = pipeline.process_customer_turn("My email is alice@example.com")
print("Auth action:", auth_res["action_taken"])
assert pipeline.case_facts.is_authenticated is True

# 2. Lookup order
order_res = pipeline.process_customer_turn("Look up ord_9901")
print("Order action:", order_res["action_taken"])
print("Trimmed Payload:", order_res["order_data"])
assert "_internal_warehouse_id" not in order_res["order_data"]

# 3. Process refund
refund_res = pipeline.process_customer_turn("Please issue the refund.")
print("Refund action:", refund_res["action_taken"])
assert refund_res["action_taken"] == "REFUND_SUCCESS"



### ⚠️ Step 3: Edge Case Testing & Guardrail Verifications
Testing boundary conditions, safety gates, error recovery, and escalation protocols.


In [ ]:
# Execute Edge Cases: Unauthenticated Refund Attempt & Immediate Escalation
pipeline2 = CustomerSupportPipeline()

# Edge Case 1: Unauthenticated lookup blocked
res_blocked = pipeline2.process_customer_turn("Please check ord_9901")
print("Blocked action:", res_blocked["action_taken"])
assert res_blocked["action_taken"] == "GATE_BLOCKED_UNAUTHENTICATED"

# Edge Case 2: Immediate human escalation
res_esc = pipeline2.process_customer_turn("I want to speak to a human right now.")
print("Escalation action:", res_esc["action_taken"])
assert res_esc["action_taken"] == "ESCALATED_TO_HUMAN"
assert pipeline2.escalated is True



### ✅ Step 4: Verification Summary & SLA Compliance
All assertions passed with zero unauthorized actions and 100% deterministic compliance.
